# Spacecraft Geometry and Self-Occlusion Demo

This notebook is the working demo for the body-fixed spacecraft geometry layer and its first self-occlusion bridge into the view-factor workflow.

What it does:
- builds the default 6U double-deployable CubeSat example
- realizes the body-fixed geometry from deployment angles
- prints a compact surface summary
- plots realized surfaces in 3D
- confirms nearest-hit ray queries on the realized geometry
- computes a patch-by-ray spacecraft self-occlusion mask
- visualizes how deployment changes the blocked part of a face's outward hemisphere
- maps the average blockage back onto the face patch grid for a panel-style view

Run the cells from top to bottom.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from geometry import build_6u_double_deployable, spacecraft_occlusion_mask


## Frame Convention

Keep the spacecraft geometry in the **body frame** by default. The builder and the realized surfaces are expressed in body coordinates, so hinge angles only change the internal geometry.

There is now one optional extra step if you want a separate geometry-frame -> body-frame attachment. Pass a rigid mount transform into `realize(...)`, for example:
- `realized = cubesat.realize(state, mount_rotation=SO3.Rz(...), mount_offset=[...])`

Use the three layers like this:
- geometry change: edit bus dimensions, hinge locations, or the deployment `state`
- mount change: edit `mount_rotation` or `mount_offset`
- attitude change: edit the `SO3` law that maps body vectors into LVLH or ECI

That keeps the structure compartmentalized: geometry answers *what the spacecraft looks like in its own frame*, the mount answers *how it is attached to the body axes*, and `SO3` answers *where the body points*.

In [ ]:
def _set_axes_equal(ax):
    x_limits = np.array(ax.get_xlim3d())
    y_limits = np.array(ax.get_ylim3d())
    z_limits = np.array(ax.get_zlim3d())

    spans = np.array([
        x_limits[1] - x_limits[0],
        y_limits[1] - y_limits[0],
        z_limits[1] - z_limits[0],
    ])
    centers = np.array([
        x_limits.mean(),
        y_limits.mean(),
        z_limits.mean(),
    ])
    radius = 0.5 * max(spans.max(), 1e-6)

    ax.set_xlim3d([centers[0] - radius, centers[0] + radius])
    ax.set_ylim3d([centers[1] - radius, centers[1] + radius])
    ax.set_zlim3d([centers[2] - radius, centers[2] + radius])


def print_surface_summary(realized):
    header = f"{'name':24s} {'center [m]':30s} {'normal':24s} {'size [m]':18s} patches tags"
    print(header)
    print('-' * len(header))
    for surface in realized.surfaces:
        center = tuple(np.round(surface.center, 4).tolist())
        normal = tuple(np.round(surface.normal, 4).tolist())
        size = f"{surface.width:.4f} x {surface.height:.4f}"
        if surface.patch_shape is None:
            patches = '1 x 1'
        else:
            patches = f"{surface.patch_shape[0]} x {surface.patch_shape[1]}"
        tags = ', '.join(surface.tags)
        print(f"{surface.name:24s} {str(center):30s} {str(normal):24s} {size:18s} {patches:7s} {tags}")


def plot_realized_geometry(realized, *, title='Spacecraft geometry', normal_scale=0.04):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    for surface in realized.surfaces:
        if 'solar_panel' in surface.tags:
            color = '#5B8FF9'
        elif 'bus' in surface.tags:
            color = '#C7C7C7'
        else:
            color = '#8C8C8C'

        corners = surface.corners()
        poly = Poly3DCollection([corners], facecolors=color, edgecolors='black', alpha=0.70)
        ax.add_collection3d(poly)

        center = surface.center
        normal = surface.normal
        ax.quiver(
            center[0], center[1], center[2],
            normal[0], normal[1], normal[2],
            length=normal_scale,
            color='crimson',
            linewidth=1.2,
        )
        ax.text(center[0], center[1], center[2], surface.name, fontsize=8)

    ax.set_title(title)
    ax.set_xlabel('body x [m]')
    ax.set_ylabel('body y [m]')
    ax.set_zlabel('body z [m]')
    ax.view_init(elev=22, azim=-58)
    _set_axes_equal(ax)
    plt.tight_layout()
    return fig, ax


def hemisphere_directions(surface, *, n_az=73, n_el=33, elevation_min_deg=5.0, elevation_max_deg=85.0):
    azimuth_deg = np.linspace(-180.0, 180.0, n_az)
    elevation_deg = np.linspace(elevation_min_deg, elevation_max_deg, n_el)
    frame = surface.frame_matrix
    dirs_body = []

    for elevation in np.radians(elevation_deg):
        ce = math.cos(elevation)
        se = math.sin(elevation)
        for azimuth in np.radians(azimuth_deg):
            local_dir = np.array([
                ce * math.cos(azimuth),
                ce * math.sin(azimuth),
                se,
            ])
            dirs_body.append(frame @ local_dir)

    return np.asarray(dirs_body), azimuth_deg, elevation_deg


def plot_occlusion_heatmap(ax, visible, azimuth_deg, elevation_deg, *, title):
    blocked = 1.0 - visible.mean(axis=0).reshape(len(elevation_deg), len(azimuth_deg))
    image = ax.imshow(
        blocked,
        extent=[azimuth_deg[0], azimuth_deg[-1], elevation_deg[0], elevation_deg[-1]],
        origin='lower',
        aspect='auto',
        vmin=0.0,
        vmax=1.0,
        cmap='magma',
    )
    ax.set_title(title)
    ax.set_xlabel('local azimuth [deg]')
    ax.set_ylabel('local elevation [deg]')
    return image, blocked


def plot_patch_occlusion_map(ax, visible, surface, *, title):
    ny, nx, _ = surface.patch_centers().shape
    blocked = 1.0 - visible.mean(axis=1).reshape(ny, nx)
    x_mm = 0.5 * surface.width * 1e3
    y_mm = 0.5 * surface.height * 1e3
    image = ax.imshow(
        blocked,
        origin='lower',
        extent=[-x_mm, x_mm, -y_mm, y_mm],
        aspect='equal',
        vmin=0.0,
        vmax=1.0,
        cmap='viridis',
    )
    x_edges = np.linspace(-x_mm, x_mm, nx + 1)
    y_edges = np.linspace(-y_mm, y_mm, ny + 1)
    ax.set_xticks(x_edges, minor=True)
    ax.set_yticks(y_edges, minor=True)
    ax.grid(which='minor', color='white', lw=0.8, alpha=0.55)
    ax.tick_params(which='minor', bottom=False, left=False)
    ax.set_title(title)
    ax.set_xlabel('surface u [mm]')
    ax.set_ylabel('surface v [mm]')
    return image, blocked


## Step 1: Build the default 6U example

Here we attach patch grids to the bus and wing surfaces. Those patch grids are not adding thickness; they just give the later occlusion and view-factor steps a patch-by-patch source mesh to work with.

In [ ]:
leaf_y = 0.2263  # panel-leaf dimension along body y [m]
leaf_z = 0.3405  # panel-leaf dimension along body z [m]
# Patch resolution only changes the sampling mesh on each surface, not the geometry itself.
bus_patch_shape = (12, 8)
wing_patch_shape = (12, 8)

cubesat = build_6u_double_deployable(
    leaf_y=leaf_y,
    leaf_z=leaf_z,
    bus_patch_shape=bus_patch_shape,
    wing_patch_shape=wing_patch_shape,
)

print('metadata:', cubesat.metadata)
print('default state keys:')
for key, value in cubesat.default_state().items():
    print(f'  {key}: {math.degrees(value):.1f} deg')


## Step 2: Realize the default deployed geometry

In [ ]:
realized_default = cubesat.realize()

print('surface count:', len(realized_default.surfaces))
print('solar panels:', [surface.name for surface in realized_default.by_tag('solar_panel')])
print()
print_surface_summary(realized_default)


In [ ]:
plot_realized_geometry(realized_default, title='Default 6U double-deployable geometry')
plt.show()


## Step 3: Try a custom deployment state

This partially folded state is kept as a geometry-viewing example only. It is useful for seeing how the builder realizes hinge angles into body-frame surfaces, but the later FOV-style diagnostics stay on the default deployment so the notebook runs faster.

In [ ]:
state_demo = {
    'wing_port_inner_angle': math.radians(70.0),
    'wing_port_outer_angle': math.radians(155.0),
    'wing_starboard_inner_angle': math.radians(-70.0),
    'wing_starboard_outer_angle': math.radians(-155.0),
}

realized_demo = cubesat.realize(state_demo)
plot_realized_geometry(realized_demo, title='Custom deployment state')
plt.show()


## Step 4: Simple nearest-hit ray check

This is still not the full view-factor propagator. It is a quick default-geometry sanity check showing that the realized spacecraft can answer the ray query the view-factor layer will need: from a source point and direction, what spacecraft surface is hit first?

In [ ]:
source_face = realized_default.by_name('bus_+X')
origin = source_face.center + 1e-4 * source_face.normal
direction = source_face.normal

hit = realized_default.first_intersection(origin, direction, exclude=('bus_+X',))
print('ray source face :', source_face.name)
print('ray origin      :', np.round(origin, 6))
print('ray direction   :', np.round(direction, 6))
print('first hit       :', None if hit is None else (hit[0].name, round(hit[1], 6)))


## Step 5: Patch-by-ray spacecraft self-occlusion

This is the new bridge into the future view-factor workflow. For a chosen source surface, we sample the outward hemisphere in the source local frame, ray-test those directions against the realized spacecraft geometry, and measure what fraction of the source patches are blocked.

To keep the demo responsive, this calculation is done only on the default deployment. The custom deployment above stays as a viewing-only geometry example.

The top panel is in direction space: each pixel is one look direction from the source face. The bottom panel is in panel space: each pixel is one face patch, colored by how often that patch is blocked across the sampled directions.

This is not yet an Earth-disk propagation. It is a geometry-only visibility diagnostic. But it is the same structural test the Earth-disk integrator will need next.

In [ ]:
source_default = realized_default.by_name('bus_+X')

dirs_default, azimuth_deg, elevation_deg = hemisphere_directions(source_default)
visible_default = spacecraft_occlusion_mask(realized_default, source_default, dirs_default)

fig, axes = plt.subplots(
    2,
    1,
    figsize=(10, 8),
    constrained_layout=True,
    gridspec_kw={'height_ratios': [1.15, 0.85]},
)

image_dir, blocked_default = plot_occlusion_heatmap(
    axes[0],
    visible_default,
    azimuth_deg,
    elevation_deg,
    title='bus_+X blocked fraction | default deployment',
)

image_patch, patch_default = plot_patch_occlusion_map(
    axes[1],
    visible_default,
    source_default,
    title='bus_+X panel view | default deployment',
)

fig.colorbar(image_dir, ax=axes[0], shrink=0.92, label='blocked patch fraction by direction')
fig.colorbar(image_patch, ax=axes[1], shrink=0.92, label='mean blocked fraction by patch')
plt.show()

print('default mean blocked fraction:', round(float(blocked_default.mean()), 3))
print('default max blocked fraction :', round(float(blocked_default.max()), 3))
print('default mean patch blockage  :', round(float(patch_default.mean()), 3))
